# 08 — D3: Heatmap probe base vs fine-tuned

Este notebook analiza los outputs del script:

```bash
scripts/run_d3_heatmap_probe.py
```

Objetivo de D3:

- Comparar BLIP base vs BLIP fine-tuneado `best/`.
- Revisar heatmaps de QK logits / cross-attention.
- Revisar heatmaps de Grad-CAM.
- Decidir si el mode collapse textual bloquea el análisis o si los mapas visuales siguen siendo reportables.

Este notebook no entrena ni extrae heatmaps directamente. Solo lee outputs, muestra figuras y ayuda a interpretar resultados.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mode_collapse_debug" / "d3_heatmaps"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Comandos D3

Cuando tengas el checkpoint fine-tuneado real, el comando principal será:

In [ ]:
cmd_real = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d3_heatmap_probe.py \\
  --ft-model-dir models/blip_finetuned_5k/best \\
  --indices data/selected_indices.json \\
  --max-images 5 \\
  --device cpu
"""

print(cmd_real)

Para verificar paths sin correr inferencia ni heatmaps:

In [ ]:
cmd_dry_run = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d3_heatmap_probe.py \\
  --ft-model-dir models/blip_finetuned_5k/best \\
  --indices data/selected_indices.json \\
  --max-images 5 \\
  --device cpu \\
  --dry-run
"""

print(cmd_dry_run)

Smoke test opcional sin Grad-CAM, útil si todavía no instalaste `grad-cam`:

In [ ]:
cmd_skip_gradcam = f"""
cd "{PROJECT_ROOT}"

python scripts/run_d3_heatmap_probe.py \\
  --allow-debug \\
  --skip-gradcam \\
  --indices data/selected_indices.json \\
  --max-images 1 \\
  --device cpu
"""

print(cmd_skip_gradcam)

## 2. Verificación de outputs

El script D3 debería generar:

```text
outputs/mode_collapse_debug/d3_heatmaps/d3_heatmap_summary.csv
outputs/mode_collapse_debug/d3_heatmaps/d3_heatmap_summary.json
outputs/mode_collapse_debug/d3_heatmaps/idx_<IDX>/base/
outputs/mode_collapse_debug/d3_heatmaps/idx_<IDX>/finetuned/
```

In [ ]:
summary_path = OUTPUT_DIR / "d3_heatmap_summary.csv"
summary_json_path = OUTPUT_DIR / "d3_heatmap_summary.json"

print("summary csv:", "OK" if summary_path.exists() else "FALTA", summary_path)
print("summary json:", "OK" if summary_json_path.exists() else "FALTA", summary_json_path)

if OUTPUT_DIR.exists():
    print("\nCarpetas de casos:")
    for p in sorted(OUTPUT_DIR.glob("idx_*")):
        print(" -", p.relative_to(PROJECT_ROOT))
else:
    print("\nTodavía no existe OUTPUT_DIR. Esto es normal si aún no corriste D3.")

In [ ]:
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print("summary_df:", summary_df.shape)
    display(summary_df)
else:
    summary_df = None
    print("Todavía no hay summary_df. Corré primero scripts/run_d3_heatmap_probe.py cuando estén los checkpoints.")

## 3. Resumen compacto

Esta tabla muestra qué salió bien o falló por modelo e imagen.

In [ ]:
if summary_df is not None:
    cols = [
        "model_tag",
        "idx",
        "cross_status",
        "gradcam_status",
        "n_cross_maps",
        "n_gradcam_maps",
        "cross_caption",
        "gradcam_caption",
        "error",
    ]
    existing_cols = [c for c in cols if c in summary_df.columns]
    display(summary_df[existing_cols])

## 4. Comparación de captions base vs fine-tuned

Sirve para ver si el modelo fine-tuneado cambió el registro lingüístico aunque los captions estén colapsados.

In [ ]:
if summary_df is not None:
    caption_cols = [
        "idx",
        "reference",
        "model_tag",
        "cross_caption",
        "gradcam_caption",
    ]
    caption_cols = [c for c in caption_cols if c in summary_df.columns]

    captions_wide = summary_df[caption_cols].copy()
    display(captions_wide)

## 5. Estado de ejecución por modelo

Esto ayuda a detectar si falló Grad-CAM, cross-attention, o ambos.

In [ ]:
if summary_df is not None:
    status_cols = ["cross_status", "gradcam_status"]
    for col in status_cols:
        if col in summary_df.columns:
            print("\n", col)
            display(summary_df.groupby(["model_tag", col]).size().reset_index(name="count"))

## 6. Galería de una imagen

Elegí un índice y este bloque muestra las figuras disponibles para BLIP base y fine-tuned.

In [ ]:
if summary_df is not None and not summary_df.empty:
    available_indices = sorted(summary_df["idx"].unique().tolist())
    print("Índices disponibles:", available_indices)

    idx_to_show = available_indices[0]
    print("idx_to_show:", idx_to_show)
else:
    idx_to_show = None

In [ ]:
def show_image_if_exists(path: Path, title: str, width: int = 900):
    if path.exists():
        display(Markdown(f"### {title}\n`{path.relative_to(PROJECT_ROOT)}`"))
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"### {title}\nFALTA: `{path.relative_to(PROJECT_ROOT)}`"))


if idx_to_show is not None:
    for model_tag in ["base", "finetuned"]:
        case_dir = OUTPUT_DIR / f"idx_{idx_to_show}" / model_tag

        display(Markdown(f"## idx={idx_to_show} — {model_tag}"))

        show_image_if_exists(case_dir / "original.png", "Imagen original", width=450)
        show_image_if_exists(case_dir / "cross_att_logits_grid.png", "QK logits / cross-attention grid")
        show_image_if_exists(case_dir / "gradcam_grid.png", "Grad-CAM grid")
        show_image_if_exists(case_dir / "cross_vs_gradcam.png", "Cross-attention vs Grad-CAM")

## 7. Galería automática de todos los casos

Muestra una vista rápida de las figuras principales para todos los índices disponibles.

In [ ]:
if summary_df is not None and not summary_df.empty:
    for idx in sorted(summary_df["idx"].unique()):
        display(Markdown(f"# Caso idx={idx}"))

        for model_tag in ["base", "finetuned"]:
            display(Markdown(f"## {model_tag}"))

            row = summary_df[
                (summary_df["idx"] == idx)
                & (summary_df["model_tag"] == model_tag)
            ]

            if not row.empty:
                r = row.iloc[0]
                print("reference:", r.get("reference", ""))
                print("cross_caption:", r.get("cross_caption", ""))
                print("gradcam_caption:", r.get("gradcam_caption", ""))
                print("cross_status:", r.get("cross_status", ""))
                print("gradcam_status:", r.get("gradcam_status", ""))
                print("error:", r.get("error", ""))

            case_dir = OUTPUT_DIR / f"idx_{idx}" / model_tag

            for fname, title in [
                ("cross_att_logits_grid.png", "QK logits / cross-attention"),
                ("gradcam_grid.png", "Grad-CAM"),
                ("cross_vs_gradcam.png", "Cross vs Grad-CAM"),
            ]:
                path = case_dir / fname
                if path.exists():
                    display(Markdown(f"### {title}"))
                    display(Image(filename=str(path), width=900))

## 8. Tabla de archivos generados

Útil para abrir imágenes manualmente desde el explorador de archivos.

In [ ]:
if summary_df is not None:
    path_cols = [
        "model_tag",
        "idx",
        "original_path",
        "cross_grid_path",
        "gradcam_grid_path",
        "comparison_path",
    ]
    path_cols = [c for c in path_cols if c in summary_df.columns]
    display(summary_df[path_cols])

## 9. Interpretación preliminar por caso

Criterio cualitativo sugerido:

- Si `base` mira zonas difusas y `finetuned` se concentra en campos pulmonares / silueta cardíaca / dispositivos, D3 apoya que el fine-tuning cambió la mirada.
- Si ambos mapas son similares o incoherentes, D3 sugiere que el cambio fue principalmente lingüístico.
- Si los captions colapsan pero los heatmaps cambian, eso es un resultado reportable: el modelo puede haber reorganizado representaciones visuales aunque el decoder no exprese bien la diversidad.

In [ ]:
if summary_df is not None:
    for idx in sorted(summary_df["idx"].unique()):
        print("=" * 120)
        print("idx:", idx)

        rows = summary_df[summary_df["idx"] == idx]

        for _, row in rows.iterrows():
            print()
            print("model:", row.get("model_tag", ""))
            print("reference:", row.get("reference", ""))
            print("cross_caption:", row.get("cross_caption", ""))
            print("gradcam_caption:", row.get("gradcam_caption", ""))
            print("cross_status:", row.get("cross_status", ""))
            print("gradcam_status:", row.get("gradcam_status", ""))
            if str(row.get("error", "")).strip():
                print("error:", row.get("error", ""))

## 10. Conclusión para informe

Completar después de revisar las figuras:

- ¿Los heatmaps fine-tuneados son visualmente distintos de los del modelo base?
- ¿El cambio parece médicamente coherente?
- ¿Cross-attention/QK logits y Grad-CAM apuntan a regiones parecidas?
- ¿El collapse textual impide el análisis, o se puede reportar como limitación metodológica?

Conclusión preliminar:

In [ ]:
conclusion = '''
Pendiente de completar después de correr D3 real.

Posibles lecturas:

1. Si captions colapsan pero heatmaps cambian:
   El fine-tuning pudo modificar la representación visual aunque el decoder produzca frases repetidas.
   Esto se reporta como hallazgo interesante y limitación del componente lingüístico.

2. Si captions mejoran y heatmaps mejoran:
   Evidencia favorable a que el fine-tuning adapta tanto lenguaje como atención visual.

3. Si captions mejoran pero heatmaps no:
   Evidencia de adaptación principalmente lingüística.

4. Si no cambia nada:
   Fine-tuning insuficiente o checkpoint demasiado débil.
'''
print(conclusion)